# Qwen on a free Colab GPU - for the MagickHome kitchen designer and its training dataset

**Only the AI model runs here.** The rule engine, catalogue, floor plans and the dataset stay on your laptop;
the laptop sends Qwen each design question and gets the answer back.

**The model (step 4):** `qwen2.5-coder:7b` (default), about 4.7 GB, 30-50 s per design. It serves both
the **training dataset** (`training/qwen_dataset.py`) and the **web app** (`colab/connect_colab.py`).
For better but slower dataset designs, set `MODEL = "qwen2.5:14b"` (about 9 GB, 1.5-2 min per design).

**Steps**
1. **Runtime -> Change runtime type -> T4 GPU -> Save.**
2. **Runtime -> Run all.** First run takes about 3-5 minutes (installs Ollama, downloads the model).
3. Copy the **command printed at the end** and run it in PowerShell on your laptop, in the project folder.
4. Keep this tab open (and the laptop awake) while it runs. When you are done: **Runtime -> Disconnect and delete runtime.**

**When a session ends** (idle or about 12 h): open the notebook again, **Run all**, and run the new command.
The dataset continues from where it stopped - nothing is lost or repeated.

**Security:** the tunnel URL is public, but every request needs the password (token) printed at the end.
Do not share it. Each new Colab session gives a new URL and a new password.

**Limits of free Colab:** sessions stop after idle time (~90 min) or ~12 h; a GPU is not always available.

## 1. Check the GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")
print("GPU:", gpu.stdout.strip())

## 2. Install Ollama

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Start Ollama (local to this Colab machine only)

In [ ]:
import os, subprocess, time, urllib.request
# NUM_PARALLEL: 4 designs at once (the dataset tool's --workers 4); the 7B model + 4 slots fit the T4
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="2h", OLLAMA_NUM_PARALLEL="4")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                               stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    print(open("ollama.log").read()[-3000:])
    raise SystemExit("Ollama did not start - its log is printed above")
print("Ollama is running")

## 4. Choose and download the model (Apache 2.0) and load it on the GPU

In [ ]:
MODEL = "qwen2.5-coder:7b"   # dataset + web app. Slower, stronger dataset designer: "qwen2.5:14b"
!ollama pull {MODEL}

In [ ]:
import json, time, urllib.request
t = time.time()
body = json.dumps({"model": MODEL, "prompt": "Reply with the single word OK.",
                   "stream": False, "options": {"num_predict": 5}}).encode()
req = urllib.request.Request("http://127.0.0.1:11434/api/generate", data=body,
                            headers={"Content-Type": "application/json"})
answer = json.load(urllib.request.urlopen(req, timeout=600))["response"].strip()
print(f"Model loaded and answered {answer!r} in {time.time() - t:.0f} s")
!ollama ps

## 5. Password proxy (only requests with the password reach Ollama; standard Python, no installs)

In [ ]:
%%writefile ollama_auth_proxy.py
"""Password (token) protected reverse proxy in front of an Ollama server - standard library only.

Runs next to Ollama on the GPU machine (Google Colab, a rented GPU, an office server).
Only requests carrying ``Authorization: Bearer <OLLAMA_API_KEY>`` reach Ollama, so a public
tunnel URL alone is not enough to use the GPU. The kitchen generator sends that header
automatically when ``OLLAMA_API_KEY`` is set on the laptop (the ``ollama`` Python client reads it).

    OLLAMA_API_KEY=<secret> python ollama_auth_proxy.py <port>

No third-party packages: works on any Python 3.8+ (including Colab's) without pip installs.
"""
import hmac
import http.client
import json
import os
import sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

UPSTREAM_HOST = os.environ.get("OLLAMA_UPSTREAM_HOST", "127.0.0.1")
UPSTREAM_PORT = int(os.environ.get("OLLAMA_UPSTREAM_PORT", "11434"))
TOKEN = os.environ["OLLAMA_API_KEY"]
DROP = {"host", "authorization", "content-length", "connection", "accept-encoding", "transfer-encoding"}


class Proxy(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def _reply(self, status: int, body: bytes, ctype: str = "application/json") -> None:
        self.send_response(status)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        if self.command != "HEAD":
            self.wfile.write(body)

    def _forward(self) -> None:
        if not hmac.compare_digest(self.headers.get("Authorization", ""), f"Bearer {TOKEN}"):
            return self._reply(401, b'{"error":"unauthorized"}')
        length = int(self.headers.get("Content-Length") or 0)
        data = self.rfile.read(length) if length else None
        headers = {k: v for k, v in self.headers.items() if k.lower() not in DROP}
        try:
            conn = http.client.HTTPConnection(UPSTREAM_HOST, UPSTREAM_PORT, timeout=None)
            conn.request(self.command, self.path, body=data, headers=headers)
            resp = conn.getresponse()
        except OSError as exc:
            return self._reply(502, json.dumps({"error": f"ollama not reachable: {exc}"}).encode())
        # Relay the answer as it is produced (chunked): a streamed design keeps the tunnel busy, so the
        # free tunnel's ~100 s limit on a silent request does not cut long answers off.
        self.send_response(resp.status)
        self.send_header("Content-Type", resp.getheader("Content-Type", "application/json"))
        self.send_header("Transfer-Encoding", "chunked")
        self.end_headers()
        try:
            while self.command != "HEAD":
                chunk = resp.read1(65536) if hasattr(resp, "read1") else resp.read(65536)
                if not chunk:
                    break
                self.wfile.write(b"%x\r\n%s\r\n" % (len(chunk), chunk))
                self.wfile.flush()
            if self.command != "HEAD":
                self.wfile.write(b"0\r\n\r\n")
        except OSError:
            pass  # the laptop went away mid-answer
        finally:
            conn.close()

    do_GET = do_POST = do_DELETE = do_HEAD = _forward

    def log_message(self, fmt, *args):  # one short line per request
        sys.stderr.write("%s %s\n" % (self.address_string(), fmt % args))


if __name__ == "__main__":
    port = int(sys.argv[1]) if len(sys.argv) > 1 else 11435
    print(f"Ollama password proxy on 127.0.0.1:{port} -> {UPSTREAM_HOST}:{UPSTREAM_PORT}", flush=True)
    ThreadingHTTPServer(("127.0.0.1", port), Proxy).serve_forever()

In [ ]:
import os, secrets, socket, subprocess, sys, time, urllib.error, urllib.request
try:
    proxy_proc.kill()  # re-running this cell: stop the previous proxy first
except NameError:
    pass
with socket.socket() as s:  # a free port (Colab uses some ports, e.g. 8080, for itself)
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]
TOKEN = secrets.token_urlsafe(24)
proxy_proc = subprocess.Popen([sys.executable, "ollama_auth_proxy.py", str(PORT)],
                              env=dict(os.environ, OLLAMA_API_KEY=TOKEN),
                              stdout=open("proxy.log", "w"), stderr=subprocess.STDOUT)
alive = False
for _ in range(20):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/tags", timeout=3)
    except urllib.error.HTTPError as e:
        alive = e.code == 401  # answering, and refusing requests without the password
        break
    except Exception:
        if proxy_proc.poll() is not None:
            break
if not alive:
    print(open("proxy.log").read()[-3000:])
    raise SystemExit("Proxy did not start - its log is printed above")
print(f"Password proxy running on port {PORT}")

## 6. Public tunnel and the commands for your laptop

In [ ]:
import json, re, subprocess, time, urllib.request
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
try:
    tunnel_proc.kill()  # re-running this cell: stop the previous tunnel first
except NameError:
    pass
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                               stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(1)
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if found:
        URL = found.group(0)
        break
if not URL:
    print(open("tunnel.log").read()[-3000:])
    raise SystemExit("Tunnel did not start - its log is printed above")

models = None
for _ in range(30):  # a brand-new tunnel URL can take a few seconds to resolve
    try:
        req = urllib.request.Request(URL + "/api/tags", headers={"Authorization": f"Bearer {TOKEN}"})
        models = [m["name"] for m in json.load(urllib.request.urlopen(req, timeout=15))["models"]]
        break
    except Exception:
        time.sleep(3)
print("Tunnel check:", "OK - models " + ", ".join(models) if models else "not answering yet (try the laptop step anyway)")
print()
print("On your laptop, in the project folder, run:")
print()
print("Training dataset (Qwen designs it; continues where it stopped):")
print()
print(f"python training/qwen_dataset.py --url {URL} --token {TOKEN} --model {MODEL} --n 50 --workers 4")
print()
print("Web app (Qwen runs here):")
print()
print(f"python colab/connect_colab.py {URL} {TOKEN}" if MODEL == "qwen2.5-coder:7b"
      else f"set OLLAMA_MODEL={MODEL} and run: python colab/connect_colab.py {URL} {TOKEN}")

## 7. Status (run any time to check everything is still up)

In [ ]:
for name, proc in (("ollama", ollama_proc), ("proxy", proxy_proc), ("tunnel", tunnel_proc)):
    print(f"{name:7s}", "running" if proc.poll() is None else f"STOPPED (exit {proc.returncode})")
!ollama ps
!tail -n 3 ollama.log

## When you are done
**Runtime -> Disconnect and delete runtime.** This stops the GPU, the model and the tunnel.

Next time: open this notebook, choose the T4 GPU, **Run all**, and run the new command on your laptop
(the URL and password change every session).

Note: the password proxy streams Qwen's answer as it is written, so long 14B designs are not cut off by the
free tunnel's ~100 s limit.